# Crypto.Net · 06 — Cosmos SDK

*Dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*

> **Aman dijalankan.** Notebook ini hanya membaca jaringan publik dan menandatangani secara offline; tidak pernah mem-broadcast transaksi. Kunci yang dibuat di sini adalah kunci demo — jangan pernah mengirim dana sungguhan ke sana.

Buka di VS Code dengan ekstensi **Polyglot Notebooks** (atau Jupyter dengan .NET Interactive) lalu jalankan sel secara berurutan. [English version](./06-cosmos.en.ipynb)

Akun Bech32 lintas chain Cosmos, pembacaan REST live, transaksi protobuf dalam `SIGN_MODE_DIRECT` dan simulasi gas oleh node sungguhan.

In [ ]:
#r "nuget: Crypto.Net.Extensions, 1.0.0"

In [ ]:
using Crypto.Net.Core;
using Crypto.Net.Native;
using Crypto.Net.Wallet;
using Crypto.Net.Cosmos;

// The public BIP-39 test phrase has an account on Cosmos Hub, which lets us run a real simulation below.
const string TestPhrase = "abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon about";
var wallet  = HdWallet.FromMnemonic(TestPhrase);
var account = wallet.GetCosmosAccount(0, CosmosChain.CosmosHub);   // m/44'/118'/0'/0/0, like Keplr
account.Address.Value

## Satu kunci, banyak chain

Kunci yang sama punya alamat di setiap chain Cosmos yang memakai coin type 118; hanya prefix Bech32 yang berubah.

In [ ]:
new[] { "cosmos", "osmo", "celestia", "juno" }
    .Select(hrp => new { Prefix = hrp, Address = CosmosAddress.ConvertPrefix(account.Address.Value, hrp) })

## Data live

In [ ]:
var rest = new CosmosRestClient(CosmosChain.CosmosHub);
var (accountNumber, sequence) = await rest.GetAccountAsync(account.Address.Value);
Console.WriteLine($"Cosmos Hub height {await rest.GetBlockNumberAsync():N0}");
Console.WriteLine($"account number {accountNumber}, sequence {sequence}");
Console.WriteLine($"balance {await rest.GetBalanceAsync(account.Address)} ATOM");

## Membangun dan menandatangani (SIGN_MODE_DIRECT)

Kirim bank dan delegasi dalam satu transaksi. Tanda tangan mencakup SHA-256 dari `SignDoc` protobuf (body, auth info, chain id, account number).

In [ ]:
var builder = new CosmosTxBuilder { Memo = "Crypto.Net notebook", GasLimit = 250_000 }
    .Add(CosmosMessage.BankSend(account.Address.Value, account.Address.Value, new Coin("uatom", 1)))
    .Add(CosmosMessage.Delegate(account.Address.Value, "cosmosvaloper1tflk30mq5vgqjdly92kkhhq3raev2hnz6eete3", new Coin("uatom", 1)));
builder.Fee.Add(new Coin("uatom", 6_250));

byte[] txRaw = account.Sign(builder, "cosmoshub-4", accountNumber, sequence);
byte[] digest = CryptoNative.Sha256(CosmosTxBuilder.EncodeSignDoc(builder.EncodeBody(), builder.EncodeAuthInfo(account.PublicKey.Span, sequence), "cosmoshub-4", accountNumber));
Console.WriteLine($"{txRaw.Length} bytes, tx hash {CosmosTxBuilder.Hash(txRaw)}");
Console.WriteLine($"signature verifies: {CryptoNative.Secp256k1Verify(account.PublicKey.Span, digest, txRaw[^64..])}");

## Simulasi gas oleh node sungguhan

Node men-decode dan mengeksekusi transaksi tanpa menyimpannya. Hasilnya membuktikan encoding protobuf sudah benar. Tidak ada yang di-broadcast.

In [ ]:
try
{
    ulong gas = await rest.SimulateAsync(txRaw);
    Console.WriteLine($"gas used {gas:N0} → a safe limit is {(ulong)(gas * 1.3):N0}");
}
catch (RpcException e) { Console.WriteLine($"node rejected the simulation: {e.RpcMessage}"); }

## Mengirim sungguhan

`SignAndBroadcastAsync` mensimulasikan, menentukan gas (×1,3), membayar gas price chain, menandatangani dan mem-broadcast:

```csharp
var testnet = wallet.GetCosmosAccount(0, CosmosChain.CosmosHubTestnet);
await using var client = new CosmosRestClient(CosmosChain.CosmosHubTestnet);
TxHash hash = await client.TransferAsync(testnet, "cosmos1…", Amount.FromAtom(0.1m), memo: "terima kasih");
```

In [ ]:
account.Dispose(); rest.Dispose(); wallet.Dispose();
"disposed"